In [2]:
import scanpy as sc
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import os
import scipy.sparse as sp

In [ ]:
sample_list_st = [
    "P1","P2","P3","P4","P5","P6","P7","P8","P9","P10","P11","P12","P13","P14","P15","P16","P26","P27","P30","P32","P33","P34","P35","P36","P37","P38","P39",
    "P40_1","P40_2","P42","P43_1","P43_2","P44","P45","P46","P47","P48","P49"
]

# GSVA gene filter

In [ ]:
def GSVA_gene_filter_sparse(adata, celltype_col, Exp=0.1, Pct=0.01, layer="counts"):

    X = adata.layers[layer] if layer in adata.layers else adata.X
    if not sp.issparse(X):
        X = sp.csr_matrix(X)
    
    celltype = np.array(adata.obs[celltype_col])
    genes = np.array(adata.var_names)
    unique_types = np.unique(celltype)

    n_genes = X.shape[1]
    meanExp_list = []
    expPct_list = []

    # --- Calculate separately for each celltype group ---
    for ct in unique_types:
        idx = np.where(celltype == ct)[0]
        subX = X[idx, :]

        mean_exp = np.array(subX.expm1().mean(axis=0)).flatten()
        mean_exp = np.log2(mean_exp + 1)
        meanExp_list.append(mean_exp)

        pct_exp = np.array((subX > 0).mean(axis=0)).flatten()
        expPct_list.append(pct_exp)

    meanExpData = np.vstack(meanExp_list)
    expPercentData = np.vstack(expPct_list)

    meanExpData = pd.DataFrame(meanExpData.T, index=genes, columns=unique_types)
    expPercentData = pd.DataFrame(expPercentData.T, index=genes, columns=unique_types)

    saveGeneByMean = meanExpData.index[meanExpData.mean(axis=1) >= Exp]

    saveGeneByExpPercent = expPercentData.loc[saveGeneByMean].index[
        expPercentData.loc[saveGeneByMean].mean(axis=1) >= Pct
    ]

    return list(saveGeneByExpPercent)

In [2]:
epi = sc.read_mtx("E:/Projects/BRCA/results/ST/SuperCell/metacell_sparse_matrix.mtx")

In [5]:
genes = pd.read_csv("E:/Projects/BRCA/results/ST/SuperCell/metacell_genes.txt", header=None, sep='\t')[0].values

In [6]:
metacells = pd.read_csv("E:/Projects/BRCA/results/ST/SuperCell/metacell_ids.txt", header=None, sep='\t')[0].values

In [7]:
epi.obs_names = metacells
epi.var_names = genes

In [8]:
sc.write('E:/Projects/BRCA/Intermediate_Data/ST/epi_objects_unified/epi_metacell.h5ad', epi)

In [ ]:
for spID in sample_list_st:
    sp_path = "E:/Projects/BRCA/results/ST/SuperCell/" + spID + "/"
    pattern = os.path.join(sp_path, "*_member.csv")
    matching_files = glob.glob(pattern)
    
    for fname in matching_files:
        basename = os.path.basename(fname)
        prefix = f"{spID}_"
        suffix = "_member.csv"
        roi = basename[len(prefix):-len(suffix)]
        member_info = pd.read_csv(fname, index_col = 0)
        member_info['swed'] = spID + "_" + roi
        member_info = member_info[['meta_cell', 'swed']]
        member_info = member_info.drop_duplicates()
        metacell_swed_info_all = pd.concat([metacell_swed_info_all, member_info], axis = 0)

In [78]:
metacell_swed_info_all.to_csv('E:/Projects/BRCA/results/ST/SuperCell/metacell_swed_info_all.csv')

In [79]:
metacell_swed_info_all.set_index('meta_cell', inplace=True)

In [91]:
saveGeneByExpPercent = GSVA_gene_filter_sparse(epi, celltype_col="swed", Exp=0.03, Pct=0.005)

d:\software\anaconda3\envs\stardist\Lib\site-packages\scipy\sparse\_data.py:149: RuntimeWarning: overflow encountered in expm1
  result = op(self._deduped_data())


In [94]:
sc.pp.normalize_total(epi, target_sum=1e4)
sc.pp.log1p(epi)

In [96]:
Counter(epi.var_names.isin(saveGeneByExpPercent))

Counter({True: 11428, False: 6657})

In [97]:
epi = epi[:, epi.var_names.isin(saveGeneByExpPercent)]

In [98]:
epi.shape

(222203, 11428)

In [99]:
X_dense = epi.X.toarray()

In [ ]:
normalized_data = pd.DataFrame(X_dense.T, index = epi.var_names, columns = epi.obs_names)
normalized_data = normalized_data.reset_index().rename(columns={'index': 'Gene'})
normalized_data.to_csv('E:/Projects/BRCA/results/ST/GSVA/metacell_gsva_filtered_input.txt', sep = '\t', index=False)